# Registros (`struct`) — Tutorial

**Programação C (COMP0512) — DCOMP/UFS — 2026.2**

Este tutorial continua de onde a aula parou: as **tarefas de modificação** e o **desafio**,
mais exercícios de prática sobre cada tópico.

## Objetivos

Ao final deste tutorial você será capaz de:

- Declarar um registro, criar variáveis dele e acessar seus campos com o operador ponto;
- Inicializar registros por lista, por campos nomeados e campo a campo;
- Compor registros dentro de registros e acessar campos encadeados;
- Explicar por que `sizeof` de um registro costuma ser maior que a soma dos campos;
- Medir o layout de um registro com `sizeof` e `offsetof` e reduzir o padding reordenando campos.

## Como usar

Cada célula de código grava um arquivo `.c` (com `%%writefile`) ou o compila e executa
(com `!gcc`). Rode as células **em ordem**. Antes de executar qualquer célula marcada
com **Preveja**, escreva sua previsão — só depois execute e compare.

In [ ]:
# Confirme que o compilador está disponível neste ambiente
!gcc --version | head -1

## 1. O programa da aula

Recompile e execute o programa condutor **exatamente como saiu da aula** e confirme
que a saída bate com a que vimos em sala.

In [ ]:
%%writefile ficha_aluno.c
#include <stdio.h>
#include <string.h>

struct Data {
    int dia;
    int mes;
    int ano;
};

struct Aluno {
    char        nome[20];
    int         matricula;
    struct Data nascimento;
    double      media;
};

int main(void) {
    struct Aluno a;
    strcpy(a.nome, "Ana Souza");
    a.matricula = 202600123;
    a.nascimento.dia = 7;
    a.nascimento.mes = 3;
    a.nascimento.ano = 2005;
    a.media = 8.75;

    printf("%s (%d)\n", a.nome, a.matricula);
    printf("Nascimento: %02d/%02d/%d\n",
           a.nascimento.dia, a.nascimento.mes, a.nascimento.ano);
    printf("Media: %.2f\n", a.media);
    printf("sizeof(struct Aluno) = %zu\n", sizeof(struct Aluno));
    return 0;
}

In [ ]:
!gcc -Wall ficha_aluno.c -o ficha_aluno && ./ficha_aluno

## 2. Investigar: onde estão os bytes que ninguém pediu?

A soma dos campos é `20 + 4 + 12 + 8 = 44`, mas `sizeof` respondeu `48`.
A ferramenta para investigar é `offsetof` (de `<stddef.h>`): ela diz a que distância
do início do registro cada campo começa.

**Preveja** os quatro offsets antes de executar a próxima célula.

In [ ]:
%%writefile layout.c
#include <stdio.h>
#include <stddef.h>   /* offsetof */

struct Data { int dia; int mes; int ano; };

struct Aluno {
    char        nome[20];
    int         matricula;
    struct Data nascimento;
    double      media;
};

int main(void) {
    printf("sizeof(struct Data)  = %zu\n", sizeof(struct Data));
    printf("sizeof(struct Aluno) = %zu\n", sizeof(struct Aluno));
    printf("--- offsets ---\n");
    printf("nome       @ %zu\n", offsetof(struct Aluno, nome));
    printf("matricula  @ %zu\n", offsetof(struct Aluno, matricula));
    printf("nascimento @ %zu\n", offsetof(struct Aluno, nascimento));
    printf("media      @ %zu\n", offsetof(struct Aluno, media));
    return 0;
}

In [ ]:
!gcc -Wall layout.c -o layout && ./layout

### Confira o raciocínio

Com os offsets em mãos, responda (por escrito, aqui mesmo nesta célula ou no caderno):

1. `nascimento` termina em qual byte? Em que byte `media` começa? Quantos bytes ficaram vazios entre eles?
2. Por que `media` não pôde começar logo depois de `nascimento`?
3. `sizeof(struct Aluno)` é múltiplo de qual número? Por que precisa ser?

_Suas respostas:_

1.
2.
3.

## 3. Tarefa 1 — um campo `char` no meio

Acrescente `char turma;` a `struct Aluno`, **logo depois de `matricula`**.

**Preveja** o novo `sizeof` antes de compilar. Depois meça e explique a diferença
(ou a ausência dela).

> Minha previsão: `sizeof(struct Aluno) = ____`

In [ ]:
%%writefile tarefa1.c
#include <stdio.h>
#include <stddef.h>

struct Data { int dia; int mes; int ano; };

struct Aluno {
    char        nome[20];
    int         matricula;
    /* TODO: acrescente aqui o campo  char turma;  */
    struct Data nascimento;
    double      media;
};

int main(void) {
    printf("sizeof = %zu\n", sizeof(struct Aluno));
    printf("nome       @ %zu\n", offsetof(struct Aluno, nome));
    printf("matricula  @ %zu\n", offsetof(struct Aluno, matricula));
    /* TODO: imprima tambem o offset de turma */
    printf("nascimento @ %zu\n", offsetof(struct Aluno, nascimento));
    printf("media      @ %zu\n", offsetof(struct Aluno, media));
    return 0;
}

In [ ]:
!gcc -Wall tarefa1.c -o tarefa1 && ./tarefa1

**Investigue:** o campo novo tem 1 byte. O tamanho total aumentou 1 byte?
Onde foi parar a diferença? Compare os offsets com os da Seção 2.

_Sua explicação:_

## 4. Tarefa 2 — reordenar do maior alinhamento para o menor

**Parte (a).** Reordene os campos de `struct Aluno` (com o `char turma` da Tarefa 1)
começando pelos de maior alinhamento: `double`, depois `int` e `struct Data`, e por
último os `char`. Meça `sizeof` antes e depois.

> Minha previsão do ganho: ____ bytes

In [ ]:
%%writefile tarefa2a.c
#include <stdio.h>
#include <stddef.h>

struct Data { int dia; int mes; int ano; };

/* ordem original da aula, com o campo turma da Tarefa 1 */
struct AlunoOriginal {
    char        nome[20];
    int         matricula;
    char        turma;
    struct Data nascimento;
    double      media;
};

struct AlunoReordenado {
    /* TODO: os MESMOS cinco campos, do maior alinhamento para o menor.
       Alinhamentos: double = 8, int = 4, struct Data = 4, char[20] = 1, char = 1 */
};

int main(void) {
    size_t soma = 20 + 4 + 1 + 12 + 8;   /* nome, matricula, turma, nascimento, media */
    printf("soma dos campos = %zu\n", soma);
    printf("original    = %zu (padding %zu)\n",
           sizeof(struct AlunoOriginal), sizeof(struct AlunoOriginal) - soma);
    printf("reordenado  = %zu (padding %zu)\n",
           sizeof(struct AlunoReordenado), sizeof(struct AlunoReordenado) - soma);
    return 0;
}

In [ ]:
!gcc -Wall tarefa2a.c -o tarefa2a && ./tarefa2a

**Surpresa?** Se o ganho foi zero, a regra não falhou --- ela simplesmente não tinha
o que economizar aqui. Investigue: quanto padding sobrava na ordem original? Um
`char[20]` seguido de um `int` já deixa o offset em 24, que é múltiplo de 4 e de 8.
Quando o desperdício é pequeno, reordenar não devolve nada.

**A regra vale a pena quando o desperdício é grande.** Parte (b): reordene
`struct Registro` abaixo e meça de novo.

> Minha previsão do ganho: ____ bytes

In [ ]:
%%writefile tarefa2b.c
#include <stdio.h>

struct Registro {
    char   flag;
    double valor;
    int    id;
    char   tipo;
};

struct RegistroBom {
    /* TODO: os mesmos quatro campos, do maior alinhamento para o menor */
};

int main(void) {
    size_t soma = 1 + 8 + 4 + 1;
    printf("soma dos campos = %zu\n", soma);
    printf("Registro    = %zu (padding %zu)\n",
           sizeof(struct Registro), sizeof(struct Registro) - soma);
    printf("RegistroBom = %zu (padding %zu)\n",
           sizeof(struct RegistroBom), sizeof(struct RegistroBom) - soma);
    printf("em um vetor de 1 milhao: economia de %zu bytes\n",
           (sizeof(struct Registro) - sizeof(struct RegistroBom)) * 1000000);
    return 0;
}

In [ ]:
!gcc -Wall tarefa2b.c -o tarefa2b && ./tarefa2b
# Meta da parte (b): sair de 24 para 16 bytes por registro.

## 5. Tarefa 3 — mais um nível de aninhamento

Crie `struct Endereco` com `char rua[30]`, `int numero` e `char cep[9]`, e inclua-a
em `struct Aluno`. Preencha e imprima `a.endereco.cep`; depois recalcule o tamanho total.

In [ ]:
%%writefile tarefa3.c
#include <stdio.h>
#include <string.h>
#include <stddef.h>

struct Data { int dia; int mes; int ano; };

/* TODO: declare struct Endereco { char rua[30]; int numero; char cep[9]; }; */

struct Aluno {
    char        nome[20];
    int         matricula;
    struct Data nascimento;
    /* TODO: acrescente o campo  struct Endereco endereco;  */
    double      media;
};

int main(void) {
    struct Aluno a = {0};
    strcpy(a.nome, "Ana Souza");
    /* TODO: preencha a.endereco.rua, a.endereco.numero e a.endereco.cep */

    printf("%s\n", a.nome);
    /* TODO: imprima  a.endereco.rua, a.endereco.numero  e  a.endereco.cep */

    printf("sizeof(struct Aluno) = %zu\n", sizeof(struct Aluno));
    /* TODO: imprima tambem sizeof(struct Endereco) e offsetof(struct Aluno, endereco) */
    return 0;
}

In [ ]:
!gcc -Wall tarefa3.c -o tarefa3 && ./tarefa3

## 6. Prática: declaração e acesso

Complete o programa abaixo. Ele deve compilar sem avisos e imprimir exatamente:

```
Ana Souza | 202600123 | 8.75
Bia Rocha | 202600124 | 6.50
```

In [ ]:
%%writefile pratica1.c
#include <stdio.h>
#include <string.h>

struct Aluno {
    char   nome[20];
    int    matricula;
    double media;
};

/* TODO: implemente a funcao imprime, que recebe um struct Aluno por copia
   e imprime uma linha no formato:  nome | matricula | media
   (nome sem alinhamento extra, media com 2 casas decimais)          */
void imprime(struct Aluno x) {
    /* TODO */
}

int main(void) {
    /* inicializacao por lista */
    struct Aluno a = {"Ana Souza", 202600123, 8.75};

    /* TODO: declare  b  e preencha campo a campo com
       "Bia Rocha", 202600124 e 6.50 (use strcpy para o nome) */
    struct Aluno b;

    imprime(a);
    imprime(b);
    return 0;
}

In [ ]:
!gcc -Wall pratica1.c -o pratica1 && ./pratica1 > saida1.txt; cat saida1.txt
!printf 'Ana Souza | 202600123 | 8.75\nBia Rocha | 202600124 | 6.50\n' > esperado1.txt
!diff -q saida1.txt esperado1.txt && echo 'OK' || echo 'Saida diferente do esperado — compare acima'

## 7. Prática: registros aninhados

Complete a função `idade_em`, que recebe duas datas e devolve a idade em anos completos.
Atenção ao caso em que o aniversário ainda não chegou no ano de referência.

In [ ]:
%%writefile pratica2.c
#include <stdio.h>

struct Data { int dia; int mes; int ano; };

struct Aluno {
    char        nome[20];
    struct Data nascimento;
};

/* TODO: devolva a idade em anos completos de quem nasceu em nasc,
   na data de referencia ref */
int idade_em(struct Data nasc, struct Data ref) {
    /* TODO */
    return 0;
}

int main(void) {
    struct Data hoje = {17, 8, 2026};
    struct Aluno a = {"Ana Souza", {7, 3, 2005}};    /* ja fez aniversario */
    struct Aluno b = {"Bia Rocha", {12, 11, 2004}};  /* ainda nao fez      */

    printf("%d\n", idade_em(a.nascimento, hoje));   /* esperado: 21 */
    printf("%d\n", idade_em(b.nascimento, hoje));   /* esperado: 21 */
    return 0;
}

In [ ]:
!gcc -Wall pratica2.c -o pratica2 && ./pratica2 > saida2.txt; cat saida2.txt
!printf '21\n21\n' > esperado2.txt
!diff -q saida2.txt esperado2.txt && echo 'OK' || echo 'Revise idade_em: um dos dois casos esta errado'

## 8. Prática: prever o `sizeof`

Para cada registro abaixo, **escreva sua previsão** de `sizeof` antes de executar.
Só depois rode a célula.

| Registro | Minha previsão | Medido |
|---|---|---|
| `struct A { char c; int i; char d; };` | | |
| `struct B { int i; char c; char d; };` | | |
| `struct C { char c; double d; int i; };` | | |
| `struct D { double d; int i; char c; };` | | |

In [ ]:
%%writefile pratica3.c
#include <stdio.h>

struct A { char c; int i; char d; };
struct B { int i; char c; char d; };
struct C { char c; double d; int i; };
struct D { double d; int i; char c; };

int main(void) {
    printf("A = %zu\n", sizeof(struct A));
    printf("B = %zu\n", sizeof(struct B));
    printf("C = %zu\n", sizeof(struct C));
    printf("D = %zu\n", sizeof(struct D));
    return 0;
}

In [ ]:
!gcc -Wall pratica3.c -o pratica3 && ./pratica3

**Investigue:** `C` e `D` têm exatamente os mesmos campos. Por que os tamanhos diferem?
Desenhe o mapa de bytes dos dois, marcando onde cada campo começa e onde entra padding.

_Sua explicação:_

## 9. Desafio — cadastro da turma

Escreva um programa que:

1. declare `struct Aluno turma[5]`, com `struct Data nascimento` aninhada;
2. preencha os cinco alunos no próprio código (sem `scanf`);
3. imprima uma tabela alinhada com nome, matrícula, data de nascimento (`dd/mm/aaaa`) e média;
4. informe o aluno de maior média e a média da turma;
5. imprima, ao final: `sizeof(struct Aluno)`, a soma dos campos, o padding **por aluno**
   e o padding **do vetor inteiro**.

Depois de funcionar, teste se reordenar os campos reduz o padding deste registro ---
e explique o resultado, seja ele qual for.

In [ ]:
%%writefile desafio.c
#include <stdio.h>
#include <string.h>

struct Data { int dia; int mes; int ano; };

struct Aluno {
    char        nome[20];
    int         matricula;
    struct Data nascimento;
    double      media;
};

/* TODO: imprima uma linha da tabela */
void imprime_linha(struct Aluno x) {
    /* TODO */
}

int main(void) {
    struct Aluno turma[5] = {
        /* TODO: preencha os cinco alunos */
    };

    /* TODO: imprima o cabecalho e as cinco linhas da tabela */

    /* TODO: encontre o aluno de maior media e calcule a media da turma */

    /* TODO: relatorio de memoria
       size_t soma = 20 + 4 + 12 + 8;
       printf("sizeof            = %zu\n", sizeof(struct Aluno));
       printf("soma dos campos   = %zu\n", soma);
       printf("padding por aluno = %zu\n", sizeof(struct Aluno) - soma);
       printf("padding no vetor  = %zu\n", (sizeof(struct Aluno) - soma) * 5);
    */
    return 0;
}

In [ ]:
!gcc -Wall desafio.c -o desafio && ./desafio

## Referências

- KERNIGHAN, B. W.; RITCHIE, D. M. *C: a linguagem de programação — padrão ANSI*. Campus, 1989. (Capítulo 6: Estruturas)
- BACKES, A. R. *Linguagem C: completa e descomplicada*. Elsevier, 2013.
- BRYANT, R. E.; O'HALLARON, D. R. *Computer Systems: A Programmer's Perspective*. 3. ed. Pearson, 2015. (Seção 3.9: alinhamento de dados)

A lista completa está em `../referencias.bib`.